## Distributed Systems Introduction

L'aumento della potenza dei computer, la capacità di memoria e archiviazione nonché la velocità delle reti hanno reso possibile la creazione di servizi utilizzati da milioni di utenti. Tali servizi devono essere capaci di produrre ed elaborare quantità enormi di dati. 

La difficoltà principale è la **scalabilità**: un servizio che funziona bene per 100 utenti potrebbe invece non funzionare per 100 milioni. Poiché una singola macchina ha limiti fisici in termini di CPU, memoria e capacità della rete per ovviare al problema è necessario distribuire e organizzare il carico di lavoro su più macchine.

La **Legge di Metcalfe** afferma che il valore di una rete cresce proporzionalmente al quadrato del numero di utenti connessi. Questo implica che più utenti si connettono a un servizio, più il servizio diventa utile e prezioso. 

Il **Gartner hype cycle** afferma che le tecnologie emergenti attraversano diverse fasi: inizialmente c'è un picco di aspettative, seguito da una fase di disillusione, e infine una fase di maturità in cui la tecnologia diventa stabile e ampiamente adottata. Il cloud computing ne rappresenta un esempio evidente.

Distribuire significa dividere qualcosa in più parti assegnandone ognuna ad entità diverse: per questo nel calcolo distribuito si applica spesso il principio di **divide et impera** (un lavoro grande per essere risolto efficaciemente deve essere diviso in più parti che possono essere elaborate in parallelo, per poi riaggregare i risultati).


### Definizione, motivazioni, caratteristiche e challenges

Per definizione (Tanenbaum et van Steen), un **Sistema Distribuito** è una **collezione di elementi di calcolo autonomi che appare agli utenti come un unico sistema coerente**. Ironicamente quindi l'utente potrebbe trovarsi a non poter usare il servizio a causa del guasto di una macchina di cui egli non sapeva nemmeno l'esistenza. In questo senso si richiama l'attenzione alla **tolleranza ai guasti** che approfondiremo poi.

**Perché costruire un sistema distribuito?**
- Condividere risorse
- Ridurre i costi: usare più macchine economiche invece di una estremamente costosa
- Migliorare le prestazioni: una richiesta può essere gestita da un nodo vicino all'utente invece che da un server centrale lontano, oppure il lavoro può essere parallelizzato
- Migliorare disponibilità e affidabilità: se un nodo si guasta, altri nodi possono continuare a fornire il servizio
- Risolvere problemi grandi: alcuni problemi richiedono più risorse di quelle disponibili su un singolo nodo, quindi è necessario distribuire il lavoro su più nodi.
- Supportare QoS (Quality of Service) e QoE (Quality of Experience): un sistema distribuito può essere progettato per garantire determinati livelli di prestazioni e affidabilità, migliorando l'esperienza dell'utente finale.

Un sistema distribuito è difficile da costruire e verificare rispetto ad uno centralizzato in quanto le sue parti agiscono separatamente e comunicano attraverso una rete. In particolare si evidenziano tre caratteristiche:
1. **Concorrenza**: più attività avvengono nello stesso periodo, es. due utenti vogliono affittare la stessa stanza di hotel nello stesso momento o due servizi vogliono modificare lo stesso dato. Occorre stabilire come coordinare queste operazioni affinché il risultato sia corretto;
2. **Assenza di orologio globale**: ogni macchina ha un proprio orologio fisico che non è sempre sincronizzato con quello delle altre macchine. Quindi non basta sempre guardare le ore registrate dai nodi per capire quale evento è avvenuto prima, questo è importante quando studieremo l'ordine degli eventi e gli orologi logici;
3. **Guasti indipendenti e parziali**: un nodo può guastarsi in qualsiasi momento, oppure può interrompersi la comunicazione tra alcuni nodi. Si parla di **guasto parziale**. Non ricevere una risposta dal server quindi non ci dice se questo sia spento, la rete abbia un problema o semplicemente la risposta sia lenta.

Le "challenge" da affrontare nella progettazione di un sistema distribuito includono:
1. **Heterogeneity**: i nodi possono avere hardware, sistemi operativi e protocolli di comunicazione diversi.
2. **Distribution Transparency**: il sistema deve nascondere la complessità della distribuzione agli utenti e ai programmatori, facendo apparire il sistema come un unico insieme coerente.
3. **Openness**: il sistema deve essere progettato in modo da poter essere esteso e modificato facilmente, senza dover riscrivere l'intero sistema.
4. **Scalability**
5. **Dependability**: il sistema deve essere affidabile e tollerante ai guasti
6. **Security**: il sistema deve proteggere i dati e le comunicazioni da accessi non autorizzati e attacchi malevoli

#### Eterogeneità e Middleware
**Eterogeneità** significa che i componenti possono essere diversi tra loro: reti, hardware, sistemi operativi, linguaggi di programmazione e implementazioni possono variare. Ad esempio un servizio scritto in Java su Linux potrebbe dover comunicare con un altro scritto in un altro linguaggio diverso, su Windows.

Il **Middleware** è uno strato software che offre funzionalità comuni ai componenti distribuiti. Sta al di sopra dei sistemi operativi e fornisce agli sviluppatori un modo uniforme di programmare e comunicare. Ovviamente il middleware non elimina fisicamente le differenze tra le macchine, ma offre astrazioni che permettono alle applicazioni di gestire queste differenze senza dover reimplementare tutto da zero. 

Alcune forme di middleware per la comunicazione che vedremo:
- **RPC (Remote Procedure Call)**: un programma invoca un'operazione eseguita da un altro nodo
- **MOM (Message Oriented Middleware)**: le componenti comunicano tramite messaggi, che possono essere inviati e ricevuti in modo asincrono

Vediamo brevemente il funzionamento di RPC tramite un esempio. Un negozio chiama  ```processPayment(card, 3.99, GBP)```. Tuttavia la funzione che elabora davvero il pagamento si trova su un altro nodo, presso il servizio di pagamento.  
Mentre la chiamata appare nel codice simile ad una funzione normale, dietro le quinte avviene quanto segue:
1. Il lato client prepara la richiesta e serializza gli argomenti (i.e. li rappresenta in modo adatto per la trasmissione sulla rete). Si parla di **marshalling**;
2. La richiesta viene inviata attraverso la rete;
3. Raggiunto il server, questo interpreta gli argomenti (**unmarshalling**) e invoca la vera implementazione di processPayment;
4. Il server analizza il risultato e lo rimanda al client, che lo interpreta.

Distinzione essenziale: una RPC assomiglia a una chiamata locale nell’interfaccia del programma, ma richiede la rete e un altro nodo. Perciò può subire ritardi o fallire per motivi che una normale chiamata locale non avrebbe.

#### Transparency
La **trasparenza** è la proprietà per cui la presenza di più nodi e la collocazione delle risorse avvengono, per quanto possibile, nascoste ad utenti e applicazioni. Si vuole far percepire il sistema come insieme coerente.

Si distinguono vari tipi di trasparenza:
- **Access transparency**: nasconde le differenze tra risorse locali e remote, permettendo di accedere a entrambe nello stesso modo
- **Location transparency**: nasconde la posizione fisica delle risorse
- **Migration transparency**: nasconde che una risorsa può spostarsi da un nodo all'altro
- **Relocation transparency**: nasconde lo spostamento di una risorsa **mentre la si sta usando**
- **Replication transparency**: nasconde il fatto che esistano più copie della stessa risorsa
- **Concurrency transparency**: permette a più utenti di accedere contemporaneamente a una risorsa senza interferenze
- **Failure transparency**: cercare di nascondere il guasto e ripristino di una componente

#### Openness
Un sistema distribuito **aperto** offre componenti che altri utenti possono utilizzare e riesce ad integrare compoonenti sviluppate altrove. Per farlo le componenti devono esporre **interfacce ben definite**: bisogna sapere quali operazioni si possono chiamare e come sono rappresentati i dati scambiati.

Si introducono in questo senso le **IDL (Interface Definition Language)**, descrizioni di strutture dati e interfacce indipendenti, per quanto possibile, da uno specifico linguaggio di programmazione (es. Protobuf, XDR, Thrift). 

I benefici cercati sono **Interoperabilità** (sistemi diversi possono comunicare), **Portabilità** (un componente può essere eseguito su piattaforme diverse) e **Estensibilità** (si possono aggiungere o sostituire componenti).

Un sistema è più facile da estendee se è composto da parti relativamente piccole, che si possono adattare o sostituire. Per progettarlo è utile distinguere tra **Meccanismo** (come il sistema rende possibile un'operazione) e **Politica** (quale scelta compie usando quel meccanismo). Un esempio è la cache del browser: un meccanismo permette di conservare copie locali di dati; le politiche invece stabiliscono quali dati conservare, quando aggiornarli e quali eliminare se lo spazio finisce.

#### Scalabilità
Un sistema è **scalabile** se riesce a mantenere le prestazioni adeguate anche quando il carico cresce significativamente. Si distinguono tre tipi di scalabilità:
- **Size Scalability** se è il numero di utenti o risorse a crescere
- **Geographical Scalability** se aumenta la distanza tra nodi
- **Administrative Scalability** se aumenta il numero di amministratori o entità che gestiscono il sistema

La scalabilità è uno dei core dei sistemi distribuiti, infatti in un sistema centralizzato una singola macchina ha limiti fisici in termini di CPU, memoria e capacità della rete. Inoltre le richieste in eccesso aspetterebbero in coda.

Definiamo ora tre grandezze: $S = \text{tempo di servizio}$ (i.e. tempo impiegato ad elaborare una richiesta), $U = \text{utilizzazione}$ (i.e. frazione di tempo in cui la macchina è occupata) e $R = \text{tempo di risposta}$ (comprende il tempo di elaborazione e il tempo di attesa in coda).  

Allora per un semplice modello di coda si ha 
$$\frac{R}{S} = \frac{1}{1-U}$$
In questo modo, se $U$ è basso, allora $R$ è vicino a $S$: la richiesta aspetta poco. Quando invece $U$ si avvicina a 1, la coda e il tempo di risposta crescono rapidamente. Ad esempio con $U = 0.9$ allora si ottiene $R = 10S$. Ciò dimostra come lavorare costantemente vicino alla saturazione è problematico.

In generale ci sono due direzioni principali per aumentare la capacità:
- **Vertical Scaling** (scale up): si rende più potente un singolo nodo, ad esempio aggiungendo CPU, RAM o storage. Tuttavia questo approccio ha limiti fisici e può essere costoso.
- **Horizontal Scaling** (scale out): si aggiungono più nodi al sistema, distribuendo il carico di lavoro tra di essi.

Le risorse possono essere fisiche ma anche potenzialmente virtuali, come macchine virtuali o container. Lo scale out è il caso dei sistemi distribuiti ma ovviamente richiede di decidere come dividere il lavoro.

In generale, per dimostrare che un sistema scala servono un parametro che cresce, una metrica misurabile e risultati sperimentali, non basta dire che usa più server.

Tra le principali tecniche per scalare:
1. **Nascondere la latenza**: con la comunicazione asincrona, un programma invia una richiesta e può continuare altro lavoro mentre aspetta la risposta. Tuttavia non tutti i modelli sono adatti a questo approccio, ad esempio se un programma deve fare una richiesta e poi immediatamente usare il risultato per continuare l'elaborazione.
2. **Spostare il calcolo**: una parte del lavoro può essere svolta dal client o da altri nodi, ad es. una decisione può essere presa vicino ai dispositivi che producono i dati oppure affidata a risorse cloud in base alle esigenze
3. **Distribuire dati e calcolo**: Si partizionano dati o compiti; di nuovo principio divide et impera.
4. **Replicare risorse e dati**: si creano più copie; una replica vicina può ridurre il tempo di accesso ed inoltre se un nodo si guasta, altre copie possono continuare a fornire il servizio. 

Tuttavia la replicazione introduce il problema per cui se una copia viene modificata, le altre copie devono essere aggiornate. Questo richiede un protocollo di **consenso** tra le repliche per garantire che tutte abbiano lo stesso stato. Inoltre dipende in base al dominio applicativo: es. per sistemi di controllo di traffico aereo si hanno requisiti molto severi. I diversi **modelli di consistenza** formalizzano queste scelte e li vedremo più avanti.

#### Dependability
Indica quanto possiamo fare affidamento sul fatto che un sistema si comporti come previsto. Nel caso dei sistemi distribuiti è una questione delicata visto che i guasti possono essere parziali e indipendenti. Si distinguono quattro proprietà:
1. **Availability**: posso usare il servizio **adesso**?
2. **Reliability**: continuerà a funzionare **senza interruzioni** per il tempo che mi serve?
3. **Safety**: quanto è improbabile che un guasto produca conseguenze catastrofiche?
4. **Maintainability**: quanto è facile ripristinare il servizio dopo un guasto?

**Availability** è la probabilità che il sistema sia operativo in un dato momento:
$$A = \frac{\text{MTTF}}{\text{MTTF} + \text{MTTR}}$$
dove MTTF (Mean Time To Failure) è il tempo medio tra guasti e MTTR (Mean Time To Repair) è il tempo medio per riparare un guasto. Si parla spesso di "numero di nove": 99% sono due nove, 99,99% sono quattro nove. La differenza pratica anche se non sembra è grande: nell'arco di un anno 99% di disponibilità è circa 3.65 giorni di inattività, mentre 99,99% sono circa 52 minuti.

La **Reliability** $R(t)$ riguarda invece la probabilità che il componente, funzionante all'inizio, continui a funzionare senza guasti per tutto l'intervallo considerato. Disponibilità e Affidabilità non coincidono quando il sistema può essere riparato. In particolare:
- un sistema che si interrompe per un ms ogni ora è disponibile per quasi tutto il tempo, ma ha guasti molto frequenti quindi è poco affidabile
- al contrario un sistema che funziona senza guasti per tutto l'anno eccetto due settimane in cui è spento ha poca disponibilità ma è molto affidabile.

Tre termini fondamentali in Dependability sono parte di una possibile catena:  
**Fault** (causa) → **Error** (stato interno errato) → **Failure** (servizio non conforme).  
Ad esempio una memoria danneggiata (fault) può produrre un valore sbagliato al suo interno (error) per cui il programma restituirà un risultato errato all'utente finale (failure). Un errore interno può causare una failure, ma non è detti che accada se viene rilevato e risolto prima che il servizio venga compromesso. 

I fault possono essere transitori, intermittenti o permanenti, e possono causare guasti a cascata tra componenti.

Per affrontare i guasti fault prevention, fault tolerance (progettare il sistema affinché continui a funzionare anche in presenza di guasti), fault removal e fault forecasting (prevedere guasti futuri e pianificare la manutenzione).

#### Sicurezza
Un sistema distribuito deve proteggere dati e operazioni scambiati tra componenti. Le slide introducono due obiettivi:
- **Confidentiality**: le informazioni sono accessibili soltanto a soggetti autorizzati
- **Integrity**: dati e risorse possono essere modificati soltanto in modo autorizzato

Si distinguono inoltre Autenticazione, Autorizzazione (verificare quali operazioni quell'identità può compiere) e Trust (avere garanzia che un'entità si comporterà come previsto). Un utente ad es. può essere autenticato correttamente ma non autorizzato a leggere un certo file.

Alcuni strumenti crittografici per la sicurezza sono crittografia simmetrica (una chiave per cifrare e decifrare), crittografia asimmetrica (una chiave pubblica per cifrare e una privata per decifrare). Si utilizzano in questo senso prevalentemente funzioni hash crittografiche (es. SHA-256) che trasformano un input di lunghezza arbitraria in un output di lunghezza fissa, in modo che sia computazionalmente difficile risalire all'input originale a partire dall'output. 

##### Esempi di sistemi distribuiti
- **Cluster Computing**: un gruppo di macchine collegate tramite una rete locale e gestite in modo coordinato. Comune nel calcolo ad alte prestazioni e domina la classifica top500 architectures. 
- **Cloud Computing**: un fornitore fornisce infrastruttura, piattaforme o software. Gli utenti ottengono servizi **su richiesta** e tipicamente pagano in base all'uso (pay-as-you-go). 
- **Edge Computing**: mentre con il cloud computing i dati vengono elaborati in data center remoti (nuvole...), con l'edge computing il calcolo viene portato verso i margini della rete, vicino a chi produce o usa i dati. Tale vicinanza aiuta quando la latenza e quantità di dati trasmessi contano, ma ha lo svantaggio che le macchine utilizzate per l'elaborazione sono spesso meno potenti e affidabili di quelle dei data center.

**Otto assunzioni sbagliate nei sistemi distribuiti**:
1. La rete è affidabile (collegamenti e messaggi non possono fallire)
2. La latenza è zero (anche alla velocità della luce ci vuole tempo per trasmettere un messaggio)
3. La banda è infinita
4. La rete è sicura
5. La topologia della rete non cambia
6. Esiste un solo amministratore (parti del sistema possono appartenere a organizzazioni diverse)
7. Trasportare dati non costa nulla
8. La rete non è omogenea

*